# LLM-RecFusion — Stage 2: Baseline ALS Recall (MovieLens-1M 真实数据)

**目标**: 使用 MovieLens-1M 真实评分数据训练 ALS 模型，为每个测试用户生成 Top-50 召回结果。

**工业级数据流 (全局一致)**:
```
data/raw/ml-1m/ratings.dat (原始 .dat :: 分隔符)
  → 按 UserID + Timestamp 全局排序
  → Leave-One-Out 策略：每个用户最后 1 条交互 → Test Set
  → 之前所有交互 → Train Set
  → 保存数据快照 (供 04_qwen 模块共享)
  → 构建 User-Item 稀疏矩阵 (CSR)
  → 训练 ALS 模型
  → 为所有 ~6040 个测试用户生成 Top-50 召回
  → 保存到 data/results/als_recall_ml_1m.json
```

In [3]:
# ============================================================
# Cell 1: 导入 & 全局配置 【完善修复版】
# 解决 pipeline 导入报错 + 鲁棒性增强 + 配置标准化
# ============================================================
import sys
import warnings
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from tqdm.auto import tqdm

# ===================== 核心修复：解决 ModuleNotFoundError: pipeline =====================
# 自动将项目根目录加入Python搜索路径，100%解决模块导入失败
notebook_path = Path.cwd()
project_root = notebook_path.parent
sys.path.insert(0, str(project_root))
# ======================================================================================

from pipeline.data_pipeline import (
    find_project_root,
    sniff_raw_data_path,
    load_ratings,
    load_movies,
    chronological_split,
    build_als_interaction_matrix,
    build_test_ground_truth,
    save_data_snapshot,
    save_recall_results,
    compute_recall_at_k,
)

# 忽略无关警告
warnings.filterwarnings("ignore")

# ---------- 路径配置 (自适应嗅探 + 鲁棒性校验) ----------
BASE_DIR = find_project_root()
RAW_DIR, RATINGS_PATH, MOVIES_PATH = sniff_raw_data_path(BASE_DIR)
RESULTS_DIR = BASE_DIR / "data" / "results"
SNAPSHOT_DIR = BASE_DIR / "data" / "snapshots"

# 自动创建目录（防止权限/路径错误）
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
SNAPSHOT_DIR.mkdir(exist_ok=True, parents=True)

# 校验原始数据文件是否存在（提前报错，避免后续运行崩溃）
assert RATINGS_PATH.exists(), f"❌ 评分文件不存在：{RATINGS_PATH}"
assert MOVIES_PATH.exists(), f"❌ 电影文件不存在：{MOVIES_PATH}"

# ---------- 运行设备配置 (ALS 加速) ----------
DEVICE = "cpu"  # 有GPU可改为 "cuda"，加速ALS训练
os.environ["MKL_NUM_THREADS"] = "8"  # CPU多线程加速

# ---------- ALS 模型超参数 ----------
ALS_FACTORS = 64            # 隐向量维度
ALS_REGULARIZATION = 0.01   # 正则化系数
ALS_ITERATIONS = 15         # 训练轮数
ALS_ALPHA = 40.0            # 隐式反馈权重
TOP_K = 50                  # 召回 Top-K 物品
N_TEST_LAST = 1             # 留一法评估：每个用户最后1条交互做测试集
RANDOM_SEED = 42            # 固定随机种子，保证实验可复现

# 全面固定随机种子（numpy + 全局）
np.random.seed(RANDOM_SEED)

# ---------- 格式化打印配置信息 ----------
print("=" * 80)
print("📂 路径配置")
print(f"✅ 项目根目录: {BASE_DIR}")
print(f"✅ 原始数据:   {RATINGS_PATH.name}")
print(f"✅ 结果保存:   {RESULTS_DIR}")
print(f"✅ 快照保存:   {SNAPSHOT_DIR}")
print("=" * 80)
print(f"⚙️  ALS 模型配置")
print(f"🔧 隐向量维度: {ALS_FACTORS} | 正则化: {ALS_REGULARIZATION} | 训练轮数: {ALS_ITERATIONS}")
print(f"🔧 召回数量: Top-{TOP_K} | 评估方式: 留一法(Last-{N_TEST_LAST}) | 设备: {DEVICE}")
print("=" * 80)

📂 路径配置
✅ 项目根目录: /home/hjy/Videos/LLM-REC/notebooks
✅ 原始数据:   ratings.dat
✅ 结果保存:   /home/hjy/Videos/LLM-REC/notebooks/data/results
✅ 快照保存:   /home/hjy/Videos/LLM-REC/notebooks/data/snapshots
⚙️  ALS 模型配置
🔧 隐向量维度: 64 | 正则化: 0.01 | 训练轮数: 15
🔧 召回数量: Top-50 | 评估方式: 留一法(Last-1) | 设备: cpu


---
## 1. 加载原始 MovieLens-1M 数据 & 严格时间序列划分

**直捣黄龙**: 直接从 `data/raw/ml-1m/ratings.dat` 读取原始数据 (:: 分隔符)。

**Leave-One-Out 划分**: 每个用户按时间戳排序后，最后 1 条交互划入 Test Set。
数据快照将持久化，供 Qwen 双塔模块共享。

In [4]:
# ============================================================
# Cell 2: 加载原始数据 & 按时间序列划分 & 保存快照
# ============================================================
print("=" * 60)
print("【1/5】加载原始 ratings.dat 并执行 Chronological Split...")
print("=" * 60)

# --- 2a. 加载原始 ratings.dat (:: 分隔符) ---
print(f"\n📂 嗅探原始数据: {RATINGS_PATH}")
ratings = load_ratings(RATINGS_PATH)
print(f"  总评分记录: {len(ratings):,}")
print(f"  唯一用户:   {ratings['UserID'].nunique():,}")
print(f"  唯一电影:   {ratings['MovieID'].nunique():,}")
print(f"  评分范围:   [{ratings['Rating'].min()}, {ratings['Rating'].max()}]")

# --- 2b. 严格按时间序列划分 (Leave-One-Out) ---
# 每个用户最后 N_TEST_LAST 条交互 → Test Set
# 之前的所有交互 → Train Set
train_df, test_df = chronological_split(
    ratings,
    strategy="loo",
    n_test=N_TEST_LAST,
)

del ratings  # 释放内存

# --- 2c. 加载 movies.dat ---
movies = load_movies(MOVIES_PATH)
print(f"\n  ✅ movies.dat 加载完成: {len(movies):,} 部电影")

# --- 2d. 保存数据快照 (供 Qwen 双塔模块共享) ---
save_data_snapshot(train_df, test_df, movies, SNAPSHOT_DIR, prefix="ml_1m")

【1/5】加载原始 ratings.dat 并执行 Chronological Split...

📂 嗅探原始数据: /home/hjy/Videos/LLM-REC/notebooks/data/raw/ml-1m/ratings.dat
  总评分记录: 1,000,209
  唯一用户:   6,040
  唯一电影:   3,706
  评分范围:   [1.0, 5.0]
  Chronological Split — 严格按时间序列划分数据
  总评分记录: 1,000,209
  唯一用户:   6,040
  唯一电影:   3,706
  时间戳范围: [956703932, 1046454590]

  策略: Leave-1-Out
  ════════════════════════════════════════
  训练集记录:    994,169  (99.4%)
  测试集记录:      6,040  (0.6%)
  ════════════════════════════════════════
  训练集唯一用户: 6,040
  训练集唯一电影: 3,704
  测试集唯一用户: 6,040
  测试集唯一电影: 1,874

  ✅ movies.dat 加载完成: 3,883 部电影

  ✅ 数据快照已保存至: /home/hjy/Videos/LLM-REC/notebooks/data/snapshots
     - 训练集: ml_1m_train.parquet (994,169 条)
     - 测试集: ml_1m_test.parquet (6,040 条)
     - 电影:   ml_1m_movies.parquet (3,883 部)
     - 信息:   ml_1m_split_info.json


---
## 2. 构建 ALS 稀疏矩阵 & 测试集 Ground Truth

使用训练集的 User-Item 交互构建 CSR 稀疏矩阵，用于 ALS 模型训练。

In [5]:
# ============================================================
# Cell 3: 构建 User-Item 稀疏矩阵 + Ground Truth
# ============================================================
print("=" * 60)
print("【2/5】构建 User-Item 稀疏矩阵 (CSR)...")
print("=" * 60)

# --- 3a. 构建 ALS 训练矩阵 ---
interaction_matrix, user2idx, item2idx, idx2item = build_als_interaction_matrix(train_df)

# --- 3b. 构建测试集 Ground Truth ---
test_user_groups = build_test_ground_truth(test_df, user2idx)
print(f"  测试用户数: {len(test_user_groups)}")

del train_df, test_df

【2/5】构建 User-Item 稀疏矩阵 (CSR)...
  Building ALS Interaction Matrix (CSR)...
  训练用户数: 6,040
  训练物品数: 3,704
  CSR 矩阵形状: (6040, 3704)
  非零元素数:   994,169
  稀疏度:       95.556223%

  Ground Truth 用户数: 6,040
  (仅保留训练集中出现过的用户)
  测试用户数: 6040


---
## 3. ALS 模型训练

使用隐式反馈 ALS (`implicit.als.AlternatingLeastSquares`) 模型。

In [6]:
# ============================================================
# Cell 4: 训练 ALS 模型
# ============================================================
print("=" * 60)
print("【3/5】训练 ALS 模型...")
print("=" * 60)

als_model = AlternatingLeastSquares(
    factors=ALS_FACTORS,
    regularization=ALS_REGULARIZATION,
    iterations=ALS_ITERATIONS,
    alpha=ALS_ALPHA,
    random_state=RANDOM_SEED,
    use_gpu=False,
)

als_model.fit(interaction_matrix, show_progress=True)

print(f"\n  ALS 训练完成!")
print(f"  User factors 形状: {als_model.user_factors.shape}")
print(f"  Item factors 形状: {als_model.item_factors.shape}")

【3/5】训练 ALS 模型...


  0%|          | 0/15 [00:00<?, ?it/s]


  ALS 训练完成!
  User factors 形状: (6040, 64)
  Item factors 形状: (3704, 64)


---
## 4. 为所有测试用户生成 Top-50 召回

遍历所有测试用户 (~6040 人)，为每个人生成 Top-50 召回结果。
使用 `tqdm.auto` 显示真实进度条。

In [7]:
# ============================================================
# Cell 5: 生成 Top-50 召回 & 持久化
# ============================================================
print("=" * 60)
print("【4/5】为测试用户生成 Top-50 召回...")
print("=" * 60)

valid_test_users = [
    uid for uid in test_user_groups
    if uid in user2idx and len(test_user_groups[uid]) > 0
]
print(f"  有效测试用户数: {len(valid_test_users):,}")

# --- 5a. 为每个测试用户生成召回列表 ---
als_recall_dict = {}

for uid in tqdm(valid_test_users, desc="ALS 召回"):
    user_idx = user2idx[uid]

    rec_indices, _ = als_model.recommend(
        user_idx,
        interaction_matrix[user_idx],
        N=TOP_K,
        filter_already_liked_items=True,
    )

    rec_items = [idx2item[idx] for idx in rec_indices]
    als_recall_dict[int(uid)] = [int(iid) for iid in rec_items]

print(f"\n  生成召回用户数: {len(als_recall_dict):,}")
print(f"  每个用户召回数: {TOP_K}")

# --- 5b. 保存为 JSON ---
als_output_path = RESULTS_DIR / "als_recall_ml_1m.json"
save_recall_results(als_recall_dict, als_output_path, "ALS", TOP_K)

【4/5】为测试用户生成 Top-50 召回...
  有效测试用户数: 6,040


ALS 召回:   0%|          | 0/6040 [00:00<?, ?it/s]


  生成召回用户数: 6,040
  每个用户召回数: 50
  ✅ ALS 召回结果已保存: /home/hjy/Videos/LLM-REC/notebooks/data/results/als_recall_ml_1m.json
  📦 文件大小: 1746.27 KB
  👤 覆盖用户数: 6,040
  🎯 Top-K: 50
  📝 预览 (用户 1 的 Top-5):
    [3159, 364, 1198, 2081, 1210]


In [8]:
# ============================================================
# Cell 6: 评估 ALS Recall@K
# ============================================================
print("=" * 60)
print("【5/5】评估 ALS Recall@K...")
print("=" * 60)

mean_recall = compute_recall_at_k(als_recall_dict, test_user_groups, TOP_K)

print(f"\n{'=' * 60}")
print(f"✅ ALS 召回模块重构完成!")
print(f"{'=' * 60}")
print(f"\n  📊 共 {len(als_recall_dict)} 个测试用户完成 Top-{TOP_K} 召回")
print(f"  📁 结果: {als_output_path}")
print(f"\n  下一步: 运行 04_qwen_dual_tower_dev.ipynb 使用相同的数据划分")
print(f"          生成 LLM 语义召回结果 → data/results/llm_recall_ml_1m.json")

【5/5】评估 ALS Recall@K...

  🔹 Recall@50 (均值): 0.2561 (25.61%)
  📊 评估用户数: 6,040

✅ ALS 召回模块重构完成!

  📊 共 6040 个测试用户完成 Top-50 召回
  📁 结果: /home/hjy/Videos/LLM-REC/notebooks/data/results/als_recall_ml_1m.json

  下一步: 运行 04_qwen_dual_tower_dev.ipynb 使用相同的数据划分
          生成 LLM 语义召回结果 → data/results/llm_recall_ml_1m.json
